# Tech Challenge Fase 2
## 03.3 — Gold Estados

Integra:

- indicadores Silver estaduais;
- metas por UF;
- indicadores da Gold Alunos por UF.

## 1. Imports

In [0]:
import json
from pathlib import Path
from datetime import datetime

import pandas as pd
import numpy as np

## 2. Configuração

In [0]:
CONFIG_FILE_PATH = "/Volumes/workspace/default/vol_trio_drive/projetos/fiap/tech_challenge_fase2/config/config.json"

config = json.loads(
    Path(CONFIG_FILE_PATH).read_text(
        encoding="utf-8"
    )
)

BASE_PATH = Path(config["environment"]["base_path"])
SILVER_PATH = Path(config["paths"]["silver_path"])
GOLD_PATH = Path(config["paths"]["gold_path"])
LOG_PATH = Path(config["paths"]["log_path"])
CONFIG_PATH = Path(config["paths"]["config_path"])
EXECUTION_DATE = config["project"]["execution_date"]

print("BASE_PATH:", BASE_PATH)
print("SILVER_PATH:", SILVER_PATH)
print("GOLD_PATH:", GOLD_PATH)
print("EXECUTION_DATE:", EXECUTION_DATE)

## 3. Funções auxiliares

In [0]:
def ler_csv(caminho, sep=";", decimal=","):
    return pd.read_csv(
        caminho,
        sep=sep,
        decimal=decimal,
        encoding="utf-8",
        low_memory=False
    )


def converter_numero(serie):
    return (
        serie
        .astype(str)
        .str.replace("%", "", regex=False)
        .str.replace(">", "", regex=False)
        .str.replace(",", ".", regex=False)
        .str.strip()
        .replace({
            "": np.nan,
            "nan": np.nan,
            "None": np.nan,
            "<NA>": np.nan,
            "-": np.nan
        })
        .pipe(pd.to_numeric, errors="coerce")
    )


def normalizar_codigo(serie):
    return (
        serie
        .astype(str)
        .str.replace(".0", "", regex=False)
        .str.strip()
    )


def salvar_csv(df, destino, nome_arquivo):
    destino = Path(destino)
    destino.mkdir(parents=True, exist_ok=True)

    df.to_csv(
        destino / nome_arquivo,
        sep=";",
        decimal=",",
        encoding="utf-8",
        index=False
    )

## 4. Leitura e integração

## Garantia da granularidade estadual

As três fontes são consolidadas previamente para uma linha por:

```text
ANO + CO_UF
```

Os joins usam validação `one_to_one`, impedindo a multiplicação silenciosa de registros.

In [0]:
metadata = pd.read_parquet(
    CONFIG_PATH / "gold_metadata"
)

df_alunos_uf = ler_csv(
    GOLD_PATH
    / "alunos_ufs"
    / "GOLD_ALUNOS_UFS.csv"
)

bases = []

def consolidar_por_uf(df, ano, codigo_origem, origem):
    df = df.copy()
    df["ANO"] = ano

    if codigo_origem not in df.columns:
        raise KeyError(
            f"{origem} {ano}: {codigo_origem} ausente."
        )

    if codigo_origem != "CO_UF":
        df = df.rename(
            columns={codigo_origem: "CO_UF"}
        )

    df["CO_UF"] = normalizar_codigo(
        df["CO_UF"]
    )

    df = df[
        df["CO_UF"].notna()
        & ~df["CO_UF"].astype(str).str.lower().isin(
            ["", "nan", "none", "<na>", "null"]
        )
    ].copy()

    chaves = ["ANO", "CO_UF"]

    numericas = [
        c for c in df.columns
        if c not in chaves
        and pd.api.types.is_numeric_dtype(df[c])
    ]

    descritivas = [
        c for c in df.columns
        if c not in chaves
        and c not in numericas
    ]

    agg = {c: "mean" for c in numericas}
    agg.update({c: "first" for c in descritivas})

    df = (
        df.groupby(chaves, dropna=False)
        .agg(agg)
        .reset_index()
    )

    if df.duplicated(
        subset=["ANO", "CO_UF"]
    ).any():
        raise ValueError(
            f"{origem} {ano}: duplicidade após consolidação."
        )

    return df


for ano in [2023, 2024, 2025]:
    meta_est = metadata[
        (metadata["produto"] == "gold_estados")
        & (metadata["dataset"] == "estados")
        & (metadata["ano"] == ano)
    ].iloc[0]

    meta_metas = metadata[
        (metadata["produto"] == "gold_estados")
        & (metadata["dataset"] == "metas_ufs")
        & (metadata["ano"] == ano)
    ].iloc[0]

    df_est = ler_csv(
        Path(meta_est["silver_path"])
        / meta_est["silver_file_name"]
    )

    df_metas = ler_csv(
        Path(meta_metas["silver_path"])
        / meta_metas["silver_file_name"]
    )

    df_alunos = (
        df_alunos_uf[
            df_alunos_uf["ANO"] == ano
        ]
        .copy()
    )

    codigo_metas = (
        "CO_UF"
        if "CO_UF" in df_metas.columns
        else "CD_UF"
    )

    df_est = consolidar_por_uf(
        df_est,
        ano,
        "CO_UF",
        "estados"
    )

    df_metas = consolidar_por_uf(
        df_metas,
        ano,
        codigo_metas,
        "metas_ufs"
    )

    df_alunos = consolidar_por_uf(
        df_alunos,
        ano,
        "CO_UF",
        "gold_alunos_uf"
    )

    df_metas_join = df_metas.drop(
        columns=[
            c for c in [
                "SIGLA_UF",
                "SG_UF"
            ]
            if c in df_metas.columns
        ],
        errors="ignore"
    )

    df_alunos_join = df_alunos.drop(
        columns=["SG_UF"],
        errors="ignore"
    )

    df_integrado = (
        df_est
        .merge(
            df_metas_join,
            on=["ANO", "CO_UF"],
            how="left",
            validate="one_to_one",
            suffixes=("", "_META")
        )
        .merge(
            df_alunos_join,
            on=["ANO", "CO_UF"],
            how="left",
            validate="one_to_one",
            suffixes=("", "_ALUNOS")
        )
    )

    if df_integrado.duplicated(
        subset=["ANO", "CO_UF"]
    ).any():
        raise ValueError(
            f"Gold Estados {ano}: duplicidade após joins."
        )

    bases.append(df_integrado)

df_gold_estados = pd.concat(
    bases,
    ignore_index=True
)

print(
    "Granularidade Gold Estados validada: "
    "uma linha por ANO + CO_UF."
)

## 5. Indicadores

In [0]:
for coluna in ["PC_ALUNO_ALFABETIZADO", "VL_MEDIA_LP", "META_FINAL_2030"]:
    if coluna in df_gold_estados.columns:
        df_gold_estados[coluna] = converter_numero(df_gold_estados[coluna])

df_gold_estados["gap_meta_2030"] = (
    df_gold_estados["META_FINAL_2030"]
    - df_gold_estados["PC_ALUNO_ALFABETIZADO"]
)

df_gold_estados["risco_educacional"] = pd.cut(
    df_gold_estados["PC_ALUNO_ALFABETIZADO"],
    bins=[-np.inf, 50, 70, 85, np.inf],
    labels=["Crítico", "Alto", "Médio", "Baixo"]
)

df_gold_estados["_gold_processed_at"] = datetime.now().isoformat()

## 6. Persistência

In [0]:
for ano in [2023, 2024, 2025]:
    registro = metadata[
        (metadata["produto"] == "gold_estados")
        & (metadata["dataset"] == "estados")
        & (metadata["ano"] == ano)
    ].iloc[0]

    salvar_csv(
        df_gold_estados[df_gold_estados["ANO"] == ano],
        Path(registro["gold_output_path"]),
        registro["gold_file_name"]
    )

print("Gold Estados salva com sucesso.")